### 1. Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

### 2. Define Tickers and Date Range

In [ ]:
# Define tickers and date range
tickers = ["TSLA", "BND", "SPY"]
start_date = "2015-01-01"
end_date = "2026-01-15"

# Download data
data = yf.download(tickers, start=start_date, end=end_date, progress=False)

# Check data structure
print("Data shape:", data.shape)
print("\nColumns:", data.columns)

### 3. Inspect Data

In [ ]:
# Show first few rows for TSLA
print(data['Adj Close'].head())

# Show last few rows
print(data['Adj Close'].tail())

### 4. Check Data Types and Missing Values

In [ ]:
# Check data types
print("Data types:")
print(data['Adj Close'].dtypes)

# Check for missing values
missing_values = data['Adj Close'].isnull().sum()
print("\nMissing values per asset:")
print(missing_values)

# Check total missing percentage
total_cells = data['Adj Close'].size
total_missing = data['Adj Close'].isnull().sum().sum()
missing_percentage = (total_missing / total_cells) * 100
print(f"\nTotal missing percentage: {missing_percentage:.2f}%")

### 5. Handle Missing Values

In [ ]:
# Forward fill missing values (common for financial data)
data_filled = data.fillna(method='ffill')

# Backward fill any remaining missing values
data_filled = data_filled.fillna(method='bfill')

# Verify no missing values remain
print("Missing values after filling:")
print(data_filled['Adj Close'].isnull().sum().sum())

### 6. Basic Statistics

In [ ]:
# Calculate basic statistics for adjusted close prices
adj_close = data_filled['Adj Close']

print("Basic Statistics (Adjusted Close Prices):")
print(adj_close.describe())

# Calculate daily returns
returns = adj_close.pct_change().dropna()

print("\nBasic Statistics (Daily Returns):")
print(returns.describe())

### 7. Closing Price Over Time

In [ ]:
# Plot adjusted closing prices
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, (ticker, color) in enumerate(zip(tickers, colors)):
    axes[idx].plot(adj_close.index, adj_close[ticker], color=color, linewidth=1.5)
    axes[idx].set_title(f'{ticker} - Adjusted Closing Price (2015-2026)', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Price ($)', fontsize=12)
    axes[idx].grid(True, alpha=0.3)
    
    # Add some key events (example for TSLA)
    if ticker == 'TSLA':
        # Example: Add vertical lines for major events
        events = {
            '2020-03-16': 'COVID Crash',
            '2020-11-16': 'S&P 500 Inclusion',
            '2022-01-03': '2022 Peak',
            '2023-01-03': '2023 Recovery'
        }
        
        for date_str, label in events.items():
            if date_str in adj_close.index.strftime('%Y-%m-%d'):
                date_idx = adj_close.index[adj_close.index.strftime('%Y-%m-%d') == date_str][0]
                axes[idx].axvline(x=date_idx, color='red', linestyle='--', alpha=0.5)
                axes[idx].text(date_idx, axes[idx].get_ylim()[0]*1.05, label, 
                              rotation=90, fontsize=8, color='red')

plt.tight_layout()
plt.show()

### 8. Daily Percentage Change

In [ ]:
# Plot daily returns
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

for idx, (ticker, color) in enumerate(zip(tickers, colors)):
    axes[idx].plot(returns.index, returns[ticker], color=color, linewidth=0.8, alpha=0.7)
    axes[idx].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[idx].set_title(f'{ticker} - Daily Returns (%)', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Return (%)', fontsize=12)
    axes[idx].grid(True, alpha=0.3)
    
    # Add mean line
    mean_return = returns[ticker].mean() * 100
    axes[idx].axhline(y=mean_return/100, color='red', linestyle='--', 
                      linewidth=1.5, label=f'Mean: {mean_return:.4f}%')
    axes[idx].legend()

plt.tight_layout()
plt.show()

# Plot distribution of returns
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (ticker, color) in enumerate(zip(tickers, colors)):
    axes[idx].hist(returns[ticker].dropna() * 100, bins=100, color=color, alpha=0.7, edgecolor='black')
    axes[idx].axvline(x=returns[ticker].mean() * 100, color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {returns[ticker].mean()*100:.4f}%')
    axes[idx].set_title(f'{ticker} - Distribution of Daily Returns', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Daily Return (%)', fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 9. Volatility with Rolling Statistics

In [ ]:
# Calculate rolling statistics (30-day window)
window = 30
rolling_mean = returns.rolling(window=window).mean()
rolling_std = returns.rolling(window=window).std()

# Plot rolling volatility
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

for idx, (ticker, color) in enumerate(zip(tickers, colors)):
    axes[idx].plot(rolling_std.index, rolling_std[ticker] * 100, 
                   color=color, linewidth=1.5, label=f'{window}-Day Rolling Volatility')
    axes[idx].set_title(f'{ticker} - Rolling Volatility (Standard Deviation)', 
                       fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Volatility (%)', fontsize=12)
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    
    # Highlight high volatility periods
    if ticker == 'TSLA':
        high_vol_threshold = rolling_std[ticker].quantile(0.9)
        high_vol_periods = rolling_std[ticker] > high_vol_threshold
        axes[idx].fill_between(rolling_std.index, 0, rolling_std[ticker]*100, 
                              where=high_vol_periods, color='red', alpha=0.3, 
                              label='High Volatility Periods (Top 10%)')
        axes[idx].legend()

plt.tight_layout()
plt.show()

### 10. Outlier Detection

In [ ]:
# Function to detect outliers using IQR method
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return outliers

# Detect outliers in returns
print("Outlier Detection (Daily Returns):")
for ticker in tickers:
    outliers = detect_outliers_iqr(returns[ticker].dropna())
    print(f"{ticker}: {len(outliers)} outliers ({len(outliers)/len(returns)*100:.2f}%)")
    
    if len(outliers) > 0:
        print(f"  Largest positive: {outliers.max()*100:.2f}%")
        print(f"  Largest negative: {outliers.min()*100:.2f}%")
        print(f"  Most extreme date: {outliers.abs().idxmax().strftime('%Y-%m-%d')}")
    print("-" * 40)

# Plot outliers for TSLA (most volatile)
tsla_outliers = detect_outliers_iqr(returns['TSLA'].dropna())
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(returns.index, returns['TSLA'] * 100, color='#FF6B6B', alpha=0.6, linewidth=0.8)
ax.scatter(tsla_outliers.index, tsla_outliers * 100, color='red', s=50, 
          label=f'Outliers ({len(tsla_outliers)} points)', zorder=5)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('TSLA - Daily Returns with Outliers Highlighted', fontsize=14, fontweight='bold')
ax.set_ylabel('Return (%)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### 11. Stationarity Test

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series, title=''):
    """
    Perform ADF test and print results
    """
    print(f'Augmented Dickey-Fuller Test: {title}')
    result = adfuller(series.dropna())
    
    labels = ['ADF Test Statistic', 'p-value', '# Lags Used', 'Number of Observations']
    for value, label in zip(result[:4], labels):
        print(f'{label}: {value:.6f}')
    
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value:.6f}')
    
    if result[1] <= 0.05:
        print("Result: Data is STATIONARY (reject H0)")
    else:
        print("Result: Data is NON-STATIONARY (fail to reject H0)")
    
    print("-" * 60)
    return result[1]  # Return p-value

# Test stationarity for closing prices
print("=" * 70)
print("STATIONARITY TEST - CLOSING PRICES")
print("=" * 70)
for ticker in tickers:
    p_value = adf_test(adj_close[ticker], f'{ticker} Closing Price')

# Test stationarity for returns
print("\n" + "=" * 70)
print("STATIONARITY TEST - DAILY RETURNS")
print("=" * 70)
for ticker in tickers:
    p_value = adf_test(returns[ticker], f'{ticker} Daily Returns')

### 12. Trend Analysis

In [ ]:
# Decompose trend for TSLA (most interesting)
from statsmodels.tsa.seasonal import seasonal_decompose

# Use multiplicative decomposition for price (additive for returns)
tsla_price = adj_close['TSLA'].dropna()

# Since we have daily data, we'll use a period of 252 (trading days in a year)
try:
    decomposition = seasonal_decompose(tsla_price, model='multiplicative', period=252)
    
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))
    
    decomposition.observed.plot(ax=axes[0], color='blue', linewidth=1)
    axes[0].set_ylabel('Observed', fontsize=12)
    axes[0].set_title('TSLA - Time Series Decomposition', fontsize=14, fontweight='bold')
    
    decomposition.trend.plot(ax=axes[1], color='red', linewidth=1.5)
    axes[1].set_ylabel('Trend', fontsize=12)
    
    decomposition.seasonal.plot(ax=axes[2], color='green', linewidth=1)
    axes[2].set_ylabel('Seasonal', fontsize=12)
    
    decomposition.resid.plot(ax=axes[3], color='purple', linewidth=1)
    axes[3].set_ylabel('Residual', fontsize=12)
    axes[3].set_xlabel('Date', fontsize=12)
    
    for ax in axes:
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except ValueError as e:
    print(f"Decomposition failed: {e}")
    print("Trying with additive model...")
    decomposition = seasonal_decompose(tsla_price, model='additive', period=252)

### 13. Value at Risk (VaR)

In [ ]:
# Calculate VaR at 95% confidence level
confidence_level = 0.95

print("Value at Risk (VaR) - 95% Confidence Level")
print("=" * 50)

for ticker in tickers:
    # Historical VaR
    var_historical = np.percentile(returns[ticker].dropna() * 100, (1 - confidence_level) * 100)
    
    # Parametric VaR (assuming normal distribution)
    mean_return = returns[ticker].mean() * 100
    std_return = returns[ticker].std() * 100
    var_param = mean_return + std_return * np.percentile(np.random.randn(10000), (1 - confidence_level) * 100)
    
    print(f"{ticker}:")
    print(f"  Historical VaR: {var_historical:.4f}%")
    print(f"  Parametric VaR: {var_param:.4f}%")
    print(f"  Worst daily return: {returns[ticker].min()*100:.4f}%")
    print("-" * 40)

### 14. Sharpe Ratio

In [ ]:
# Calculate annualized Sharpe Ratio
# Assuming risk-free rate = 0.02 (2%) for simplicity
risk_free_rate = 0.02
trading_days = 252

print("Sharpe Ratio (Annualized)")
print("=" * 40)

for ticker in tickers:
    excess_returns = returns[ticker] - (risk_free_rate / trading_days)
    
    # Annualize
    annualized_return = (1 + returns[ticker].mean()) ** trading_days - 1
    annualized_volatility = returns[ticker].std() * np.sqrt(trading_days)
    
    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_volatility
    
    print(f"{ticker}:")
    print(f"  Annualized Return: {annualized_return*100:.2f}%")
    print(f"  Annualized Volatility: {annualized_volatility*100:.2f}%")
    print(f"  Sharpe Ratio: {sharpe_ratio:.4f}")
    print("-" * 40)

### 15. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = returns.corr()

# Visualize correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Daily Returns', fontsize=16, fontweight='bold')
plt.show()

# Print correlation insights
print("Correlation Insights:")
print("=" * 40)
print("TSLA-SPY correlation:", correlation_matrix.loc['TSLA', 'SPY'])
print("TSLA-BND correlation:", correlation_matrix.loc['TSLA', 'BND'])
print("SPY-BND correlation:", correlation_matrix.loc['SPY', 'BND'])

### 16. Summary DataFrame

In [ ]:
# Create a summary table
summary_data = []

for ticker in tickers:
    ticker_data = {
        'Asset': ticker,
        'Start Price': adj_close[ticker].iloc[0],
        'End Price': adj_close[ticker].iloc[-1],
        'Total Return (%)': ((adj_close[ticker].iloc[-1] / adj_close[ticker].iloc[0]) - 1) * 100,
        'Annualized Return (%)': ((adj_close[ticker].iloc[-1] / adj_close[ticker].iloc[0]) ** (1/11) - 1) * 100,
        'Daily Return Mean (%)': returns[ticker].mean() * 100,
        'Daily Return Std (%)': returns[ticker].std() * 100,
        'Max Daily Gain (%)': returns[ticker].max() * 100,
        'Max Daily Loss (%)': returns[ticker].min() * 100,
        'Sharpe Ratio': ((returns[ticker].mean() * 252 - risk_free_rate) / (returns[ticker].std() * np.sqrt(252))),
        'VaR 95% (%)': np.percentile(returns[ticker].dropna() * 100, 5)
    }
    summary_data.append(ticker_data)

summary_df = pd.DataFrame(summary_data)
summary_df.set_index('Asset', inplace=True)

print("SUMMARY STATISTICS (2015-2026)")
print("=" * 70)
print(summary_df.round(4))